# Part1

In [2]:
import sentence_transformers
import transformers
import huggingface_hub
import langchain_huggingface

print(sentence_transformers.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)


3.0.1
4.46.1
0.33.4


In [5]:
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# PDF path (YOU will change this)
PDF_PATH = "C:/Users/kumar/OneDrive/Desktop/TRY-3/Tasks-Submission/Assignment34/4 - Harry Potter and the Goblet of Fire.pdf"


# Task1

In [4]:
# Load PDF document
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

# Combine all pages into one text
full_text = "\n".join([doc.page_content for doc in documents])

# Print stats
print("Total Characters:", len(full_text))
print("\nSample Preview:\n")
print(full_text[:500])


Total Characters: 1094523

Sample Preview:



Harry Potter
and the Goblet Of Fire
 
 
by
J. K. Rowling
Illustrations by Mary Grandpré
 
 
 
 
Arthur A. Levine Books
An Imprint of Scholastic Press
To Peter Rowling,
In Memory of Mr. Ridley
And to Susan Sladden,
Who helped Harry
Out of his cupboard
Text copyright © 2000 by J.K. Rowling
Illustrations by Mary GrandPre copyright © 2000 Warner
Bros.
All rights reserved. Published by Scholastic Press, a division
of Scholastic Inc.,
Publishers since 1920.
SCHOLASTIC, SCHOLASTIC PRESS, and the LANT


# Task2

In [ ]:
# Prompt template
prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are a professional document summarizer.
Summarize the following text clearly and concisely.

Text:
{text}
"""
)

# LLM Chain
prompt_chain = LLMChain(
    llm=llm,
    prompt=prompt
)

# Generate summary
prompt_summary = prompt_chain.run(full_text)

print(prompt_summary)


# Task3

In [ ]:
# Short summary prompt
short_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following text in 5–6 lines:\n{text}"
)

# Bullet summary prompt
bullet_prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following text in bullet points:\n{text}"
)

short_chain = LLMChain(llm=llm, prompt=short_prompt)
bullet_chain = LLMChain(llm=llm, prompt=bullet_prompt)

short_summary = short_chain.run(full_text)
bullet_summary = bullet_chain.run(full_text)

print("SHORT SUMMARY:\n", short_summary)
print("\n" + "-"*80 + "\n")
print("BULLET SUMMARY:\n", bullet_summary)

"""
Comparison:
- Short summary is narrative and compact
- Bullet summary is structured and easier to scan
"""


# Part2

# Task4

In [ ]:
"""
1. Stuff chain sends the entire document to the LLM at once.
2. Suitable for small to medium documents.
3. Limitations:
   - Token limits
   - Not scalable for large PDFs
"""


# Task5

In [ ]:
# Stuff summarization chain
stuff_chain = load_summarize_chain(
    llm=llm,
    chain_type="stuff"
)

stuff_summary = stuff_chain.run(documents)

print(stuff_summary)


# Task6

In [ ]:
print("PROMPT-BASED SUMMARY:\n", prompt_summary)
print("\n" + "-"*80 + "\n")
print("STUFF CHAIN SUMMARY:\n", stuff_summary)

"""
Differences:
- Prompt-based uses raw text
- Stuff chain uses LangChain document handling
- Both fail for very large documents
"""


# Part3

# Task7

In [ ]:
"""
1. Large documents exceed LLM context windows.
2. Map step summarizes chunks independently.
3. Reduce step combines chunk summaries into a final output.
"""


# Task8

In [ ]:
# Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(documents)

# Map-Reduce chain
map_reduce_chain = load_summarize_chain(
    llm=llm,
    chain_type="map_reduce"
)

map_reduce_summary = map_reduce_chain.run(split_docs)

print(map_reduce_summary)


# Task9

In [ ]:
map_chain = load_summarize_chain(
    llm=llm,
    chain_type="map_reduce",
    return_intermediate_steps=True
)

result = map_chain({"input_documents": split_docs})

for i, summary in enumerate(result["intermediate_steps"], 1):
    print(f"Chunk {i} Summary:\n{summary}\n")

print("FINAL SUMMARY:\n", result["output_text"])


# Part4

# Task10

In [ ]:
"""
1. Refine chain creates an initial summary from first chunk.
2. Each new chunk refines the existing summary.
3. Difference:
   - Map-Reduce: parallel
   - Refine: sequential and coherent
"""


# Task11

In [ ]:
refine_chain = load_summarize_chain(
    llm=llm,
    chain_type="refine"
)

refine_summary = refine_chain.run(split_docs)

print(refine_summary)


# Task12

In [ ]:
"""
Prompt-based:
- Fast
- Not scalable

Stuff:
- Good coherence
- Token limited

Map-Reduce:
- Scales best
- Slight context loss

Refine:
- Best coherence
- Slower
"""


# Part5

# Task13

In [ ]:
def summarize_document(text, method="map_reduce"):
    docs = [text]

    if method == "prompt":
        return llm.invoke("Summarize:\n" + text).content

    if method == "stuff":
        chain = load_summarize_chain(llm, chain_type="stuff")
        return chain.run(docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    split_docs = splitter.split_documents(docs)

    if method == "map_reduce":
        chain = load_summarize_chain(llm, chain_type="map_reduce")
        return chain.run(split_docs)

    if method == "refine":
        chain = load_summarize_chain(llm, chain_type="refine")
        return chain.run(split_docs)

    raise ValueError("Invalid method")


# Task14

In [ ]:
"""
1. Best for very long docs: Map-Reduce
2. Best quality: Refine
3. Trade-off:
   - Speed vs coherence
4. Real-world use:
   - Prompt: emails
   - Stuff: blogs
   - Map-Reduce: research papers
   - Refine: legal / policy documents
"""
